In [ ]:
# seismo-benchmark-control
import os as _seismo_bench_os
import sys as _seismo_bench_sys
import time as _seismo_bench_time
from pathlib import Path as _SeismoBenchPath

_seismo_bench_here = _SeismoBenchPath.cwd()
for _seismo_bench_root in [_seismo_bench_here, *_seismo_bench_here.parents]:
    if (
        (_seismo_bench_root / "benchmarking" / "seismo_bench.py").exists()
        or (_seismo_bench_root / "seismo_bench.py").exists()
    ):
        if str(_seismo_bench_root) not in _seismo_bench_sys.path:
            _seismo_bench_sys.path.insert(0, str(_seismo_bench_root))
        break

try:
    from benchmarking.seismo_bench import make_context as _seismo_bench_make_context
    from benchmarking.seismo_bench import start_run as _seismo_bench_start_run
except ModuleNotFoundError:
    from seismo_bench import make_context as _seismo_bench_make_context
    from seismo_bench import start_run as _seismo_bench_start_run

def _seismo_bench_float_from_env(name):
    value = _seismo_bench_os.environ.get(name, "").strip()
    if not value:
        return None
    try:
        return float(value)
    except ValueError:
        return None

_SEISMO_BENCH_CONTEXT = _seismo_bench_make_context(
    notebook="ObsPy/ObsPy_Basic_Example.ipynb",
    environment=_seismo_bench_os.environ.get("SEISMO_BENCH_ENV", "unset"),
    browser=_seismo_bench_os.environ.get("SEISMO_BENCH_BROWSER", "unset"),
    device=_seismo_bench_os.environ.get("SEISMO_BENCH_DEVICE", "unset"),
    run_index=int(_seismo_bench_os.environ.get("SEISMO_BENCH_RUN_INDEX", "0")),
    platform_peak_memory_mb=_seismo_bench_float_from_env("SEISMO_BENCH_PLATFORM_PEAK_MEMORY_MB"),
    notes=_seismo_bench_os.environ.get("SEISMO_BENCH_NOTES", ""),
)
_SEISMO_BENCH_RUN = _seismo_bench_start_run(_SEISMO_BENCH_CONTEXT)
_SEISMO_BENCH_TOTAL_T0 = _seismo_bench_time.perf_counter()
_SEISMO_BENCH_PENDING_PHASE = None
_SEISMO_BENCH_PENDING_T0 = None

def _seismo_bench_start_phase(phase):
    global _SEISMO_BENCH_PENDING_PHASE, _SEISMO_BENCH_PENDING_T0
    _SEISMO_BENCH_PENDING_PHASE = phase
    _SEISMO_BENCH_PENDING_T0 = _seismo_bench_time.perf_counter()

def _seismo_bench_save_silently():
    try:
        _SEISMO_BENCH_RUN.save()
    except Exception as exc:
        print(f"Benchmark save failed: {exc}")

def _seismo_bench_post_run_cell(result):
    global _SEISMO_BENCH_PENDING_PHASE, _SEISMO_BENCH_PENDING_T0
    if _SEISMO_BENCH_PENDING_PHASE is None:
        return
    if _SEISMO_BENCH_PENDING_T0 is None:
        _SEISMO_BENCH_PENDING_PHASE = None
        return
    raw_cell = getattr(getattr(result, "info", None), "raw_cell", "")
    if "seismo-benchmark-control" in raw_cell:
        return
    elapsed = _seismo_bench_time.perf_counter() - _SEISMO_BENCH_PENDING_T0
    error = getattr(result, "error_in_exec", None) or getattr(result, "error_before_exec", None)
    success = error is None and bool(getattr(result, "success", True))
    notes = ""
    if error is not None:
        notes = f"{type(error).__name__}: {error}"
    _SEISMO_BENCH_RUN.record_manual_phase(
        _SEISMO_BENCH_PENDING_PHASE,
        elapsed,
        success=success,
        notes=notes,
    )
    _SEISMO_BENCH_PENDING_PHASE = None
    _SEISMO_BENCH_PENDING_T0 = None
    _seismo_bench_save_silently()

_seismo_bench_ip = get_ipython()
try:
    _seismo_bench_old_callback = getattr(
        _seismo_bench_ip, "_seismo_bench_post_run_cell_callback", None
    )
    if _seismo_bench_old_callback is not None:
        _seismo_bench_ip.events.unregister("post_run_cell", _seismo_bench_old_callback)
except Exception:
    pass
_seismo_bench_ip.events.register("post_run_cell", _seismo_bench_post_run_cell)
_seismo_bench_ip._seismo_bench_post_run_cell_callback = _seismo_bench_post_run_cell
print("Benchmark context ready:", _SEISMO_BENCH_CONTEXT)


## Basic ObsPy Example

This is a short example of a common ObsPy use case:
 * Download waveform data of an earthquake from a data center (via FDSN web service)
 * Download the corresponding station metadata including the frequency response (via FDSN web service)
 * Remove the instrument response from the waveform data to get physical units

For more detailed examples, check out e.g.:
 * [Seismo-Live](https://seismo-live.github.io/)
 * the [ObsPy Tutorial](https://docs.obspy.org/tutorial/index.html) pages

First, just some minimal setup for the plotting and adjustments currently needed for compatibility with web assembly

In [ ]:
# seismo-benchmark-control
_seismo_bench_start_phase('setup/imports')


In [ ]:
# JupyterLite/emscripten compatibility for ObsPy file readers.
# ObsPy's MiniSEED reader uses np.memmap, which is unavailable in the browser
# filesystem. Copy file bytes into an ndarray instead.
import numpy as np

def _seismo_patch_obspy_wasm_io():
    def _seismo_fake_memmap(filename, dtype=np.uint8, mode=None, offset=0, shape=None, order=None):
        with open(filename, "rb") as f:
            f.seek(offset)
            data = np.fromfile(f, dtype=dtype)
        if shape is not None:
            data = data[:int(np.prod(shape))].reshape(shape, order=order or "C")
        return data

    np.memmap = _seismo_fake_memmap

    try:
        import pyodide_http
        pyodide_http.patch_all()
    except Exception:
        pass

_seismo_patch_obspy_wasm_io()

# this is required to monkey patch any urllib requests to use urllib3/requests.
# urllib is not WASM compatible. Emscripten tries to automatically reroute any of its download
# requests to a ws:// websocket which fails with a "Mixed content" error on an https:// deployment.

import pyodide_http
pyodide_http.patch_all()  # Patch all libraries

In [ ]:
# seismo-benchmark-control
_seismo_bench_start_phase('setup/imports')


In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
plt.style.use('ggplot')
plt.rcParams['figure.figsize'] = 12, 8

In this example we download data from the Black Forest Observatory in Germany for the 2011 Tohoku earthquake

In [ ]:
# seismo-benchmark-control
_seismo_bench_start_phase('setup/model')


In [ ]:
from obspy import UTCDateTime

network = "GR"
station = "BFO"
location = ""
channel = "LH*"
t = UTCDateTime("2011-03-11T05:46:23")  # Tohoku Earthquake
t1 = t + 10 * 60
t2 = t + 70 * 60

Now, download the waveform data from BGR server..

In [ ]:
# seismo-benchmark-control
_seismo_bench_start_phase('data loading/download')


In [ ]:
from obspy.clients.fdsn import Client

# service discovery uses multithreading which currently does not work here
# but it is not really needed here anyway, since it is mostly to discover
# custom query parameters the FDSN server might support in addition to
# the default query parameters
client = Client("BGR", _discover_services=False)

st = client.get_waveforms(network, station, location, channel, t1, t2)
print(st)
# semicolon to avoid jupyter showing the plot twice
st.plot();

These waveforms are still in arbitrary units (aka. "digital counts") and affected by the frequency response of the recording equipment. To make sense of the data in terms of actual amplitudes in physical units and also to see correct phase onset polarities (especially if close to the corner frequency), we download the corresponding station metadata including the instrument response.

In [ ]:
# seismo-benchmark-control
_seismo_bench_start_phase('data loading/download')


In [ ]:
inventory = client.get_stations(
    network=network, station=station, location=location, channel=channel,
    starttime=t1, endtime=t2, level="response")
print(inventory)
# semicolon to avoid jupyter showing the plot twice
inventory.plot_response(min_freq=0.0001);

We can use the station metadata to correct for the instrument response and show the data in physical units, here as ground velocity in meter/second.

In [ ]:
# seismo-benchmark-control
_seismo_bench_start_phase('main compute')


In [ ]:
st.remove_response(inventory, water_level=10, output="VEL")
st.plot();

In [ ]:
# seismo-benchmark-control
_SEISMO_BENCH_RUN.record_manual_phase(
    "full notebook total",
    _seismo_bench_time.perf_counter() - _SEISMO_BENCH_TOTAL_T0,
)
_seismo_bench_csv_path, _seismo_bench_json_path = _SEISMO_BENCH_RUN.save()
print("Benchmark results written:", _seismo_bench_csv_path, _seismo_bench_json_path)
